# 모델 서빙 프레임워크로 모델 서빙하기

## 1. 과정 소개

### 1.1 교과목 기획 배경
앞선 수업에서는 FastAPI나 Streamlit을 통해, 모델을 외부에서 호출 가능한 형태로 배포하는 기본 방법을 살펴봤습니다.  
즉, **사용자가 요청을 보내면 결과를 돌려주는 API나 앱을 만드는 입구**를 먼저 익힌 것입니다.

하지만 실제 서비스에서는 여기서 한 단계 더 들어가야 합니다.  
모델을 단순히 연결하는 것만으로는 부족하고, **추론 자체를 얼마나 안정적이고 효율적으로 처리할 것인가**가 중요해집니다.

예를 들어 GPU 기반 모델을 운영할 때는 다음과 같은 문제가 생깁니다.
- 모델 로딩과 재시작을 직접 관리해야 함
- 동시 요청이 늘어나면 처리량 최적화가 어려움
- 배치 처리, GPU 자원 활용, 모델 버전 관리 등을 애플리케이션 코드에서 직접 다루기 복잡함
- 일반 모델과 LLM처럼 모델 특성이 다르면 운영 방식도 달라짐

그래서 실무에서는 FastAPI나 Streamlit 같은 애플리케이션 레이어와 별도로,  
**모델 추론을 전문적으로 담당하는 서빙 프레임워크**를 사용합니다.

- Triton: CNN, ONNX, TensorRT 등 다양한 모델을 고성능으로 서빙하는 범용 추론 서버
- vLLM: LLM 추론에 특화되어 동시 요청 처리와 KV Cache 관리를 최적화한 서버

즉, 이번 수업의 초점은 배포 일반론이 아니라,  
**모델 추론 서버를 전문화하면 무엇이 달라지는지**를 이해하는 데 있습니다.

이후 실습에서는 Triton으로 일반 딥러닝 모델 서빙을, vLLM으로 LLM 서빙을 각각 다루며  
서빙 프레임워크가 왜 별도로 필요한지 비교해봅니다.


## 2. 모델 서빙 프레임워크 개요

### 2.1 왜 모델 서빙 프레임워크가 필요한가?
모델을 학습하거나 실험할 때는 보통 **한 번 실행할 코드**를 작성합니다.  
예를 들어 Python 스크립트나 노트북에서 모델을 로드한 뒤, 입력을 넣고 결과를 확인하는 방식입니다.

하지만 서비스 환경에서는 요구사항이 달라집니다.
- 모델이 항상 메모리나 GPU 위에 준비된 상태로 떠 있어야 함
- 외부 요청을 API 형태로 계속 받아야 함
- 요청 수가 시시각각 달라져도 지연 시간과 처리량을 함께 관리해야 함
- 모델 교체, 버전 관리, 재시작, 상태 확인 같은 운영 작업이 반복적으로 발생함

즉, 문제는 단순히 "모델을 실행할 수 있느냐"가 아니라,  
**모델 추론을 지속적으로 운영 가능한 서버 형태로 만들 수 있느냐**입니다.

이 차이를 메우기 위해 Triton, vLLM 같은 서빙 프레임워크를 사용합니다.

### 2.2 서빙 프레임워크의 역할
일반 PyTorch 코드만으로도 배치를 구성하거나 여러 요청을 처리하는 로직을 직접 만들 수 있습니다.  
하지만 서빙 프레임워크는 이 기능을 **애플리케이션 코드 바깥의 추론 서버 레이어**로 분리해서 제공합니다.

구체적으로는 다음 역할을 맡습니다.
- **모델 실행 표준화**: 모델을 일정한 API 형태로 노출해, 애플리케이션은 내부 구현 대신 정해진 요청 주소로 호출만 하도록 만듭니다.
- **요청 스케줄링과 배치 최적화**: 들어오는 요청을 순서대로 잠시 모아두고, 필요하면 자동으로 묶어 GPU 효율을 높입니다.
- **모델 라이프사이클 관리**: 모델 로딩, 언로딩, 버전 관리, 재시작 시 복구를 서버 차원에서 다룹니다.
- **성능 최적화 분리**: TensorRT, ONNX Runtime, KV Cache 최적화처럼 모델별 추론 최적화를 애플리케이션 코드와 분리합니다.
- **운영 관측성 제공**: readiness, health check, metrics 같은 운영용 인터페이스를 기본으로 제공합니다.

즉, 서빙 프레임워크의 차별점은 단순히 "여러 요청을 받을 수 있다"가 아니라,  
**추론 성능 최적화와 운영 관리 기능을 서버 자체가 담당한다**는 데 있습니다.  
이 점 때문에 FastAPI 같은 웹 프레임워크와도 경쟁 관계라기보다, 함께 조합되는 경우가 많습니다.

## 3. Nvidia Triton 이해하기

### 3.1 Triton의 개념과 특징

NVIDIA Triton Inference Server는 GPU 환경에서 다양한 딥러닝 모델을
효율적으로 서빙하기 위한 **추론 전용 서버**입니다.

**주요 특징**

- 다양한 프레임워크 지원 (TensorFlow, PyTorch, ONNX 등)  
- GPU 최적화를 통한 **높은 성능** (추론 속도↑, 메모리 사용량↓)  
- **Dynamic Batching 지원**:
  - 개별 요청을 즉시 처리하지 않고, 짧은 시간 동안 요청을 받아 배치 단위로 GPU에 전달
  - GPU 커널 호출 횟수 감소 -> 처리량 증가

### 3.2 Triton 설치 (Docker 기반)

Triton은 Docker 컨테이너 형태로 제공되며,
GPU 드라이버만 올바르게 설치되어 있다면 별도의 복잡한 설치 과정 없이 사용 가능합니다.

공식 이미지 목록:
https://catalog.ngc.nvidia.com/orgs/nvidia/containers/tritonserver

```bash
# 최신 이미지 (권장하지 않음: 버전 변동 위험)
docker pull nvcr.io/nvidia/tritonserver:latest

# 실습 및 수업용 고정 버전
docker pull nvcr.io/nvidia/tritonserver:25.12-py3
```

### 3.3 Triton 서버 실행 (정석 방식)

Triton Inference Server는 실행 시점에 지정된 모델 저장소를 스캔하여 모델을 로딩합니다.  
모델 구성이 잘못되었거나 파일이 누락되면 서버가 즉시 종료되므로, 정확한 디렉터리 구조와 파일 배치가 무엇보다 중요합니다.  

#### 3.3.1 모델 저장소 디렉터리 구조

Triton은 다음과 같은 **고정된 디렉터리 규칙**을 사용한다.

```text
/models/ (Root Directory)
├── resnet18/                # 모델 A (이름: resnet18)
│   ├── config.pbtxt         # 모델 설정 파일
│   └── 1/                   # 버전 관리 폴더 (숫자 필수)
│       ├── model.onnx       # 모델 실행 파일 (파일명 고정)
│       └── resnet18.onnx.data  # [선택] 외부 가중치 파일
└── bert_classifier/         # 모델 B (이름: bert_classifier)
    ├── config.pbtxt
    └── 1/
        └── model.onnx       # 가중치가 포함된 단일 파일 형식
```

#### 3.3.2 ONNX 모델의 저장 방식에 따른 파일 구성

Triton에서 ONNX 모델을 배포할 때는 해당 모델이 어떤 방식으로 저장되었는지에 따라 파일 구성을 맞춰야 합니다.

| 구분 | **단일 파일 형식 (Single File)** | **외부 데이터 분리 형식 (External Data)** |
| :--- | :--- | :--- |
| **특징** | 모델 구조와 가중치가 `model.onnx` 하나에 통합됨 | 모델 구조(`model.onnx`)와 가중치(`.data`)가 분리됨 |
| **장점** | 파일 하나만 관리하면 되어 이동 및 배포가 간편함 | 2GB 이상의 대형 모델(LLM 등) 처리에 필수적임 |
| **Triton 배치** | 버전 폴더(`1/`) 내에 `model.onnx`만 배치 | **`model.onnx`와 `.data` 파일을 반드시 같은 폴더에 배치** |

> **⚠️ 주의사항**
> ONNX 모델 파일(`model.onnx`) 내부에는 외부 가중치 파일의 이름이 참조값으로 기록되어 있습니다. 만약 원본 파일명이 `resnet18.onnx.data`였다면, Triton 폴더 내부에서도 해당 이름을 그대로 유지해야 모델 로딩에 실패하지 않습니다.

#### 3.3.3 주요 구성 요소 상세 설명

* **모델명 폴더 (`resnet18/`)**: Triton API를 통해 추론을 요청할 때 사용하는 엔드포인트 이름이 됩니다.
* **버전 폴더 (`1/`)**: 숫자로 된 폴더가 최소 하나 이상 존재해야 합니다. Triton은 기본적으로 가장 높은 숫자 버전의 모델을 활성화합니다.
* **실행 파일 (`model.onnx`)**: Triton ONNX 런타임은 파일명이 반드시 `model.onnx`여야 인식합니다. 원본 파일명이 다르더라도 이 폴더 안에서는 이름을 변경해 주어야 합니다.
* **설정 파일 (`config.pbtxt`)**: 모델의 입출력 텐서 형태, 데이터 타입, 배칭 전략 등을 정의합니다.

#### 3.3.4 서버 실행 시 모델 저장소 지정 예시

Triton 컨테이너 실행 시 호스트 시스템의 경로를 컨테이너 내부의 `/models` 경로로 마운트하고, `--model-repository` 옵션을 사용합니다.

```bash
docker run --gpus=all -p 8000:8000 -v /home/user/my_models:/models \
  nvcr.io/nvidia/tritonserver:23.10-py3 \
  tritonserver --model-repository=/models
```

#### 3.3.5 Triton 컨테이너 생성 및 실행

```bash
docker run -d \
  --name triton-server \
  --gpus all \
  -p 7000:8000 \
  -p 7001:8001 \
  -p 7002:8002 \
  -v /home/triton/model_repository:/models \
  nvcr.io/nvidia/tritonserver:25.12-py3 \
  tritonserver --model-repository=/models
```

**포트 설명**
- 7000 -> 8000 : HTTP REST API
  - 브라우저, `curl`, Python HTTP 클라이언트처럼 가장 익숙한 방식으로 Triton 서버에 요청할 때 사용하는 포트
- 7001 -> 8001 : gRPC API
  - gRPC는 Google이 만든 고성능 원격 호출(RPC) 프로토콜로, 바이너리 기반이라 HTTP REST보다 빠르고 효율적인 통신이 필요한 환경에서 자주 사용
- 7002 -> 8002 : Metrics (Prometheus)
  - Metrics는 서버 상태를 수치로 보여주는 정보로, Prometheus 같은 모니터링 도구가 이 포트에서 요청 수, 지연 시간, GPU/메모리 사용량 등을 수집할 때 사용

**중요 주의사항**
- `~`(홈 디렉터리 축약)은 사용하지 말고 **절대 경로**를 사용
- 컨테이너 생성 시 모델 로딩 실패 → 서버 즉시 종료
- 문제 발생 시 `docker start`가 아닌 **컨테이너 재생성**이 정석

#### 3.3.6 Triton 로그 확인

```bash
docker logs triton-server
```

정상 로딩 시 반드시 다음 메시지가 출력된다.

```text
successfully loaded 'resnet18'
| resnet18 | 1 | READY |
```

### 3.4 Triton 상태 확인 방법 (REST API)

```bash
# 서버 준비 상태 확인 
curl -i http://localhost:7000/v2/health/ready
```
  - HTTP/1.1 200 OK 나오면 정상
  - 404면 모델 이름/경로 문제
  - 503이면 서버/모델이 아직 준비 안 됨

```bash
# 특정 모델 준비 상태 확인
curl -i http://localhost:7000/v2/models/resnet18/ready

# 모델 설정 확인
curl http://localhost:7000/v2/models/resnet18/config
```

## 4. Triton 실습 예시

### 4.1 PyTorch → ONNX 변환 (Dynamic Batch)

실무에서는 **배치 크기 고정 모델이 아닌 Dynamic Batch 모델**을 사용하는 것이 일반적입니다.  
4-2.딥러닝 모델 변환과 양자화.ipynb 수업자료에서 만든 모델을 그대로 사용합니다. 

```python
torch.onnx.export(                     # PyTorch 모델을 ONNX 형식으로 변환
    model,                             # 변환 대상 PyTorch 모델
    sample_input,                      # 더미 입력 텐서 (shape: (1, 3, 224, 224))
    "resnet18.onnx",                   # 출력될 ONNX 모델 파일 경로
    input_names=["input"],             # ONNX 입력 텐서 이름 (input: [-1, 3, 224, 224])
    output_names=["output"],           # ONNX 출력 텐서 이름 (output: [-1, 1000])
    opset_version=18,                  # 사용할 ONNX opset 버전
    do_constant_folding=True,          # 상수 연산을 미리 계산하여 그래프 최적화
    dynamic_axes={                     # 동적 배치 크기 설정 
                                       # 배치 크기가 달라져도 하나의 배치로 묶어 처리해야하기 위해 배치 축을 알려줌
        "input": {0: "batch"},         # input의 0번 축을 batch 차원으로 설정 ([-1, 3, 224, 224])
        "output": {0: "batch"}         # output의 0번 축을 batch 차원으로 설정 ([-1, 1000])
    }
)
```

변환 결과 ONNX 입력/출력 형태:
```text
input  : [-1, 3, 224, 224]
output : [-1, 1000]
```

In [ ]:
!pip install tritonclient[http]

#### Triton 추론 요청 방식 설명

Triton Inference Server는 HTTP 기반 API를 제공하므로  
이론적으로는 `curl` 명령으로도 추론 요청을 보낼 수 있습니다.  

다만 이미지 파일을 **FP32 NCHW 텐서로 변환하여 바이너리 형태로 전송해야 하므로**,  
`curl`만으로 추론을 수행하는 것은 실습 및 개발 단계에서 매우 번거롭습니다.  

따라서 실제로는 Triton에서 제공하는 **Python client(`tritonclient`)** 를 사용해  
아래와 같이 추론 요청을 보내는 방식이 일반적입니다.  

이 예제는 **서빙 엔진(Triton)을 애플리케이션 코드에서 어떻게 호출하는지**를  
가장 단순한 형태로 보여줍니다.

In [ ]:
import numpy as np
from PIL import Image
import tritonclient.http as httpclient

client = httpclient.InferenceServerClient("localhost:7000")

img = Image.open("image/dog.jpg").convert("RGB")
img = img.resize((224,224))
img = np.array(img).astype(np.float32) / 255.0
img = img.transpose(2,0,1)
img = np.expand_dims(img, axis=0)

inputs = httpclient.InferInput("input", img.shape, "FP32")
inputs.set_data_from_numpy(img)

outputs = httpclient.InferRequestedOutput("output")

result = client.infer(
    model_name="resnet18",
    inputs=[inputs],
    outputs=[outputs]
)

print(result.as_numpy("output").argmax(axis=1))

[259]


## 5. Triton 고급 기능

### 5.1 Dynamic Batching
앞 절에서는 ONNX 모델이 **여러 batch 크기를 받을 수 있도록** `dynamic_axes`를 설정했습니다.  
이제는 한 단계 더 나아가, **서버가 들어온 요청을 실제로 어떻게 묶어서 처리할지**를 봅니다.

즉, 앞 절과 이번 절은 비슷해 보이지만 역할이 다릅니다.
- **dynamic_axes**: 모델이 `[1, ...]`, `[8, ...]` 같은 다양한 batch 크기 입력을 받을 수 있게 준비하는 단계 (Inference 가능 여부)
- **dynamic_batching**: Triton이 여러 요청을 아주 잠깐 모아 하나의 batch로 합쳐 실행하는 단계 (Scheduler의 전략)

이 구분을 먼저 잡아두면, 왜 지금 Dynamic Batching을 보는지 자연스럽게 이해할 수 있습니다.

#### 왜 필요한가?
- 서비스 환경에서는 요청이 보통 1건씩 들어오지만, GPU는 어느 정도 묶어서 처리할 때 더 효율적입니다.
- 요청을 즉시 처리하면 개별 응답은 빠를 수 있지만, 요청이 몰릴 때 전체 처리량(Throughput)은 낮아질 수 있습니다.
- 반대로 아주 짧은 시간 동안 요청을 모으면 GPU 활용률을 극대화할 수 있습니다.

#### 동작 방식
1. 클라이언트는 평소처럼 요청을 1건씩 보냅니다.
2. Triton은 내부 큐에서 요청을 아주 짧게 기다립니다.
3. **대기 시간(`delay`)이 다 되거나, 효율적인 개수(`preferred_size`)가 모이면** 하나의 batch로 합쳐 GPU에 전달합니다.
4. 실행 결과는 다시 각 요청 단위로 나누어 반환합니다.

핵심은 **클라이언트 코드를 복잡하게 만들지 않고**, **서버 쪽에서 자동으로 배치 최적화**를 수행한다는 점입니다.

**config.pbtxt 예시 (Dynamic Batching 활성화)**
```text
name: "resnet18"
platform: "onnxruntime_onnx"
max_batch_size: 32

input [
  {
    name: "input"
    data_type: TYPE_FP32
    dims: [3, 224, 224]  # 배치 차원을 제외한 데이터 자체의 shape만 적음
  }
]
output [
  {
    name: "output"
    data_type: TYPE_FP32
    dims: [1000]
  }
]

dynamic_batching {
  preferred_batch_size: [4, 8, 16]
  max_queue_delay_microseconds: 2000
}
```

#### 설정을 읽는 순서
- **max_batch_size: 32**: 하드웨어와 모델의 메모리 한계상 한 번에 최대 32개까지만 묶겠다는 **최종 상한선**입니다.

- **preferred_batch_size: [4, 8, 16]**: 대기 시간이 남았더라도 이 숫자만큼 요청이 모이는 순간 **즉시 출발**하라는 효율적 지점들입니다.  
(예: 4개가 차는 순간 2ms를 다 채우지 않고 바로 추론 시작)
- **max_queue_delay_microseconds: 2000**: 요청이 너무 적어서 `preferred_size`를 못 채우더라도,  
최대 2ms까지만 기다리고 무조건 출발하라는 **타임아웃** 설정입니다.  
- **input / output의 dims**: Triton은 `max_batch_size`가 0보다 크면 자동으로 맨 앞에 배치 차원이 있다고 가정합니다.  
따라서 여기엔 순수 데이터 형상만 적어야 하며, 배치 차원을 포함한 `[1, 3, 224, 224]`처럼 적지 않도록 주의해야 합니다.  

#### 세 가지 처리 방식 비교

| 방식 | 구현 방법 | 장점 | 단점 |
| :--- | :--- | :--- | :--- |
| **1건 즉시 처리** | 요청이 들어올 때마다 바로 `infer()` 호출 | 구조가 단순하고 개별 응답 속도가 가장 빠름 | 전체 처리량(Throughput)이 낮고 GPU 활용률이 떨어짐 |
| **클라이언트 수동 배치** | 요청을 직접 모아 큰 batch로 묶어서 전달 | Throughput이 매우 높고 대량 처리에 유리 | 클라이언트 로직이 복잡해지고 대기 시간이 발생함 |
| **Triton Dynamic Batching** | **클라이언트는 1건씩 보내고 서버가 자동 배치** | **코드 복잡도는 낮으면서 처리량은 획기적으로 개선** | `preferred_size`와 `queue delay` 사이의 정밀한 튜닝 필요 |


#### 언제 특히 유용한가?
- 요청이 꾸준히 들어오는 **실시간 API 서버** 환경에서 가장 실용적인 선택입니다.
- 사용자 경험(지연 시간)을 해치지 않으면서도 서버 비용(GPU 효율)을 아끼고 싶을 때 필수적입니다.
- 반면, 실시간성이 아예 필요 없는 **오프라인 대량 작업**이라면 클라이언트에서 수동으로 크게 묶어 보내는 것이 더 효율적일 수 있습니다.

#### 여기서 중요한 트레이드오프 (Trade-off)
- **Queue Delay를 늘리면**: 더 큰 batch를 만들 가능성이 커져 GPU 효율은 극대화되지만, 먼저 들어온 요청의 대기 시간(Latency)이 함께 늘어납니다.
- **Queue Delay를 줄이면**: 응답은 빨라지지만, 배치 효과가 줄어들어 전체적인 서버 처리 용량이 낮아질 수 있습니다.

즉, Dynamic Batching은 단순히 켜기만 하면 성능이 올라가는 옵션이 아니라, **우리 서비스의 목표 지연 시간과 목표 처리량 사이의 최적의 균형점(Sweet Spot)을 찾는 과정**입니다.

### 5.2 실무 프로젝트 의뢰 사례: 단일 GPU 서버 동시성 최적화

아래는 실제 현업에서 자주 발생하는 모델 서빙 이슈를 정리한 사례입니다.  
단일 요청은 빠른데, 동시 요청이 늘면 지연이 급격히 커지는 전형적인 상황입니다.

<img src="image/프로젝트의뢰.png">

원문 출처: https://discuss.pytorch.kr/t/tx-5090-cosyvoice-llm-concurrency/8977/1

#### 사례 요약
- 환경: RTX 5090 단일 서버, CosyVoice(TTS) 서빙, TensorRT 적용
- 문제: 단일 요청은 2초 이내 처리되지만, 동시 요청 시 Queue가 쌓여 실시간 처리 실패
- 쟁점: "GPU 독점 사용 특성상 하드웨어 추가 없이는 개선 불가" 주장의 타당성 검증 필요
- 목표: 소프트웨어 레벨에서 동시성 확보 및 처리량 개선

#### 이 사례에서 핵심 포인트
- 한 요청씩 바로 처리하면 지연은 짧아도, 요청이 몰릴 때 전체 대기시간이 길어질 수 있습니다.
- 요청을 아주 짧게 모아 한 번에 처리하면 GPU를 더 효율적으로 사용할 수 있습니다.
- API 서버가 동기 방식이면 요청이 줄 서기 쉬우므로, 비동기 처리 구조가 도움이 됩니다.
- 일반 모델/TTS 서빙은 Triton, LLM 서빙은 vLLM처럼 역할을 나누면 운영이 쉬워집니다.

#### 실무 점검 순서
1. 먼저 속도 저하 구간을 분리해서 측정합니다. (모델 추론, 전처리/후처리, 요청 대기)
2. 요청이 몰리는 시간대의 동시 요청 수를 확인합니다.
3. 서버가 여러 요청을 동시에 받도록 구성되어 있는지 확인합니다.
4. 요청을 아주 짧게 모아 한 번에 처리하도록 설정해 봅니다.
5. 변경 전/후를 비교합니다. (평균 응답시간, 느린 구간 응답시간, 초당 처리량)


## 6. vLLM을 활용한 LLM 서빙 실습

### 6.1 vLLM 개요

vLLM은 대규모 언어 모델(LLM)의 **추론 성능 최적화**를 목적으로 개발된
오픈소스 LLM 서빙 엔진입니다.

Transformer 기반 LLM의 특성을 고려하여,
GPU 메모리 사용을 최소화하면서도 높은 처리량을 제공하는 데 초점을 두고 있습니다.

**vLLM의 핵심 특징은 다음과 같습니다.**
- LLM 추론에 특화된 서빙 엔진
- OpenAI API 호환 인터페이스 제공
- 효율적인 KV Cache 관리
- 높은 동시 요청 처리 성능

### 6.2 vLLM이 필요한 이유

기존의 일반적인 LLM 서빙 방식은 다음과 같은 한계를 가집니다.

- 요청 수가 증가할수록 GPU 메모리 사용량 급증
- 각 요청마다 KV Cache를 독립적으로 관리 → 메모리 낭비
- 배치 크기 증가 시 응답 지연(latency) 발생

vLLM은 이러한 문제를 해결하기 위해 **PagedAttention**이라는 메모리 관리 기법을 사용합니다.

**PagedAttention의 개념**
- KV Cache를 연속 메모리가 아닌 페이지 단위로 관리(쪼개사용)
- 서로 다른 요청 간에 GPU 메모리 공간을 유연하게 공유
- 불필요한 메모리 재할당 감소

이를 통해 vLLM은 다음과 같은 장점을 가집니다.
- 동일 GPU 환경에서 더 많은 동시 요청 처리 가능
- 응답 지연 감소
- 대화형 서비스(Chatbot)에 적합한 성능 제공

### 6.3 Triton과 vLLM의 역할 차이

| 구분 | Triton | vLLM |
|----|----|----|
| 주 용도 | 범용 모델 추론 서버 | LLM 전용 추론 서버 |
| 지원 모델 | CNN, ONNX, TensorRT 등 | Transformer 기반 LLM |
| 추론 방식 | 정형 입력/출력 | 텍스트 생성 (Autoregressive) |
| API 형태 | REST / gRPC | OpenAI 호환 API |
| 활용 예 | 이미지 분류, 음성 인식 | 챗봇, RAG, 에이전트 |  

<br>

실무에서는 **일반 모델은 Triton**, **LLM은 vLLM**으로 역할을 분리하여 사용하는 경우가 많습니다.

### 6.4 실습 목표

이번 실습에서는 vLLM을 활용하여
로컬 환경에서 LLM 서버를 구성하고,
OpenAI API와 동일한 방식으로 모델을 호출해봅니다.

또한, 이전에 구현한 RAG 파이프라인에서
LLM 부분을 vLLM 서버 기반 모델로 교체하는 흐름을 경험합니다.

### 6.6 Gemma-3 1B 모델 사용 권한 신청

실습에서 사용할 Gemma 계열 모델은 사용 권한 승인이 필요합니다.

아래 링크에서 Hugging Face 계정으로 신청합니다.  
👉 https://huggingface.co/google/gemma-3-1b-it

- 승인까지 약 1~2시간 소요될 수 있습니다.
- 승인 후 Hugging Face 토큰을 발급받아야 합니다.

### 6.5 실습 흐름

1. Hugging Face에서 Gemma-3 1B 모델 사용 권한 신청
2. vLLM 서버 실행 및 LLM API 엔드포인트 생성
3. curl 명령어로 LLM API 호출 테스트
4. LangChain에서 OpenAI 대신 vLLM 서버 사용
5. RAG 파이프라인과 연동

### 6.7 Hugging Face 토큰 설정

```python
import os
from dotenv import load_dotenv

load_dotenv("env.txt")
hf_token = os.getenv("HF_TOKEN")

if not hf_token:
    raise ValueError("환경 변수 HF_TOKEN이 설정되어 있지 않습니다.")
```

- HF_TOKEN은 Hugging Face 계정에서 발급받은 토큰입니다.
- 토큰은 환경 변수로 관리하는 것이 권장됩니다.

### 6.8 vLLM 서버 실행

터미널에서 아래 명령어를 실행하여 vLLM 서버를 기동합니다.

```bash
vllm serve google/gemma-3-1b-it \
  --host 0.0.0.0 \
  --port 8000 \
  --cpu-offload-gb 8
```

`google/gemma-3-1b-it`은 Hugging Face 토큰이 필요한 gated 모델입니다. 토큰 없이 실행하면 모델 다운로드 단계에서 `401 Unauthorized`, `403 Forbidden`, `Cannot access gated repo` 같은 오류가 날 수 있습니다.

앞 셀에서 `HF_TOKEN`을 읽어왔더라도, 그 값은 파이썬 코드 안에서만 사용됩니다.  
터미널에서 `vllm serve`를 실행할 때는 토큰을 쉘 환경 변수로 다시 설정하거나 Hugging Face CLI 로그인을 해두어야 합니다.

```bash
export HF_TOKEN=hf_xxx

vllm serve google/gemma-3-1b-it \
  --host 0.0.0.0 \
  --port 8000 \
  --cpu-offload-gb 8
```

또는 아래처럼 로그인한 뒤 다시 실행해도 됩니다.

```bash
huggingface-cli login
```

**옵션 설명**
- `google/gemma-3-1b-it`  
  → Hugging Face 모델 이름 (사전 승인 필요)
- `--host 0.0.0.0`  
  → 외부 요청 수신 허용
- `--port 8000`  
  → API 서버 포트
- `--cpu-offload-gb`  
  → GPU 메모리가 부족한 경우 일부 KV Cache를 CPU 메모리로 오프로딩

서버 실행이 성공하면 OpenAI와 호환되는 API 엔드포인트가 자동으로 생성됩니다.

### 6.9 curl을 이용한 API 호출 테스트

```bash
curl -X POST "http://localhost:8000/v1/chat/completions" \
  -H "Content-Type: application/json" \
  --data '{
    "model": "google/gemma-3-1b-it",
    "messages": [
      {
        "role": "user",
        "content": "한국의 수도는 어디인가요?"
      }
    ]
  }'
```

정상 동작 시 모델의 응답이 JSON 형태로 반환됩니다.

### 6.10 LangChain에서 vLLM 사용 예시

vLLM은 OpenAI API와 호환되므로,
LangChain에서 OpenAI 대신 동일한 인터페이스로 사용할 수 있습니다.

```python
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model_name="google/gemma-3-1b-it",
    openai_api_base="http://localhost:8000/v1",
    openai_api_key="dummy_key",
    temperature=0.7,
    max_tokens=512,
)
```

- vLLM은 API Key 검증을 수행하지 않으므로 임의의 값 사용 가능
- 기존 LangChain 코드 변경 없이 서버 주소만 교체하여 사용 가능

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model_name="google/gemma-3-1b-it",
    openai_api_base="http://localhost:9300/v1",
    openai_api_key="dummy_key",
    temperature=0.7,
    max_tokens=512,
)

In [ ]:
from langchain_core.output_parsers import StrOutputParser  # 모델 응답 객체에서 텍스트만 꺼내는 파서입니다.
from langchain_core.prompts import PromptTemplate  # 문자열 프롬프트 템플릿을 만드는 도구입니다.
from langchain_openai import ChatOpenAI  # OpenAI 채팅 모델을 LangChain에서 호출합니다.

prompt = PromptTemplate.from_template("'{topic}'를 초보자도 이해할 수 있게 한 문장으로 설명해줘.")  # 같은 질문 형식을 계속 재사용합니다.


llm = ChatOpenAI(
    model_name="google/gemma-3-1b-it",
    openai_api_base="http://localhost:9300/v1",
    openai_api_key="dummy_key",
    temperature=0.7,
    max_tokens=512,
)

parser = StrOutputParser()  # 모델 응답에서 텍스트만 남겨 비교를 단순하게 만듭니다.

# 같은 프롬프트를 실행 방식만 바꿔 비교할 수 있도록 체인을 준비합니다.
chain = prompt | llm | parser

In [13]:
chain.invoke({"topic": "인공지능"})

'인공지능은 컴퓨터가 인간처럼 생각하고 학습하여 문제를 해결하는 기술입니다.'

### 6.11 정리

- Triton은 범용 추론 서버입니다.
- vLLM은 LLM 추론에 특화된 서버입니다.
- vLLM은 높은 동시 요청 처리와 빠른 응답에 강점을 가집니다.
- RAG, 챗봇, 에이전트 기반 시스템에서는 vLLM이 특히 효과적입니다.

## 7. 클라우드 배포 실습

### 7.1 목표
- 원하는 모델을 Triton or vllm을 사용하여 클라우드에서 각각 배포
- 가능하면 외부에서 배포한 모델을 사용해보기 (보안 및 포트 개방 등 추가 설정 필요)
- 배포 프레임워크를 사용할 때와 안할 때의 성능(추론 속도, 메모리 사용량) 비교  

### 7.2 발표
- 팀별로 결과 공유 
- 어떤 프레임워크가 더 적합했는지 토론